In [7]:
!pip install pandas kaggle trl liger-kernel accelerate unsloth
# !kaggle competitions download -c deep-past-initiative-machine-translation
# !unzip deep-past-initiative-machine-translation.zip -d deep_past_data_past_data

import pandas as pd
train_df = pd.read_csv("deep_past_data_past_data/train.csv")
test_df = pd.read_csv("deep_past_data_past_data/test.csv")

In [8]:
train_df

,oare_id,transliteration,translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...
...,...,...,...
1556,ff3208e4-8ab8-4368-b4df-7b80afa5bc32,um-ma en-nam-a-šur-ma a-na a-la-ḫi-im qí-bi-ma...,From Ennam-Aššur to Ali-ahum: Here 2 men have ...
1557,ff43a284-3d67-4238-8b4a-9b6cb7531e0a,3 ma-na KÙ.BABBAR ṣa-ru-pá-am i-na ší-im SÍG.Ḫ...,Ilī-ašrannī son of Sukkalliya has received 3 m...
1558,ff5747a4-af8a-4100-a906-a2660ae72606,ša-lim-a-šùr a-na a-mur-IŠTAR ú-ṭá-ḫi-ni-a-tí-...,Šalim-Aššur made us approach Amur-Ištar and Ša...
1559,ff777871-97ce-4bfc-bdfb-73352868944d,a-na en-nam-a-šùr qí-bi-ma um-ma IŠTAR-ra-bi₄-...,To Ennam-Aššur from Ištar-rabiʾat: With respec...


In [9]:
def create_text_column(row):
    if "translation" not in row or pd.isna(row["translation"]):
        return "Akkadian:" + row["transliteration"] + ":" + "English:"
    return "Akkadian:" + row["transliteration"] + ":" + "English: " + row["translation"]

In [10]:
train_df["text"] = train_df.apply(create_text_column, axis=1)
test_df["text"] = test_df.apply(create_text_column, axis=1)

validation_df = train_df.sample(frac=0.1, random_state=0)
train_df = train_df.drop(validation_df.index)

from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
validation_dataset = Dataset.from_pandas(validation_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
from unsloth import FastLanguageModel
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig
import torch

def train_worker():
    # Safe in spawned processes; will crash if the launcher uses fork.
    bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "LiquidAI/LFM2.5-1.2B-Base",
        dtype = torch.bfloat16 if bf16_ok else torch.float16,
        trust_remote_code = True,
        max_seq_length = 512,
        load_in_4bit = True,
        full_finetuning = False, # Focus on LoRA layers only
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = 32
        , # This controls the rank of the LoRA update. Increase for larger models/datasets for better quality.
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj",],
        lora_alpha = 512, # This controls the scaling of the LoRA update. Usually set to r*16.
        lora_dropout = 0, # Dropout for LoRA layers.
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 0,
        use_rslora = True,
        loftq_config = None,
    )

    training_args = SFTConfig(
        output_dir="akkadian-translation-model",
        dataset_text_field="text",
        report_to = "none", # Use TrackIO/WandB etc
        num_train_epochs=1,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        args=training_args,
    )

    trainer.train()
    trainer.save_model(training_args.output_dir)

train_worker()

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.1.2: Fast Qwen3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.254 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Tokenizing ["text"] (num_proc=36): 100%|██████████| 156/156 [00:06<00:00, 25.24 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,405 | Num Epochs = 1 | Total steps = 176
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Step,Training Loss
10,2.878100
20,2.150700
30,1.914400
40,1.781900
50,1.622200
60,1.608300
70,1.560900
80,1.490700
90,1.556300
100,1.441300


In [15]:
# Load the model and tokenizer using Unsloth for inference
model_path = "akkadian-translation-model"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 512,
    dtype = None, # Auto-detect
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# Function to generate translation from the text column using the optimized model
def generate_translation(text):
    inputs = tokenizer([text], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 100, use_cache = True)
    # Decode only the generated part by skipping the prompt tokens
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return generated_text.strip()

# Generate predictions
# We use the 'text' field which contains the prompt format defined in create_text_column
debug_df = train_df.head(10).copy()

def remove_translation_part(text):
    # Remove the "English: " part to only keep the Akkadian transliteration
    if "English:" in text:
        return text.split("English:")[0] + "English:"
    return text

debug_df["text"] = debug_df["text"].apply(remove_translation_part)

debug_df["predicted_translation"] = debug_df["text"].apply(generate_translation)
debug_df["id"] = debug_df.index

# Save result
submission_df = debug_df[["id", "predicted_translation"]]
submission_df.to_csv("akkadian_translations.csv", index=False)

out = debug_df[["transliteration", "translation", "predicted_translation"]]
out.to_csv("akkadian_debug_output.csv", index=False)

out

==((====))==  Unsloth 2026.1.2: Fast Qwen3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.254 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Input IDs of shape torch.Size([1, 546]) with length 546 > the model's max sequence length of 512.
We shall truncate it ourselves. It's imperative if you correct this issue first.


,transliteration,translation,predicted_translation
0,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...","Seal of Mannum-bālum-Aššur son of Šiladum, sea..."
1,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...,"1 good textile, Itūr-ilī has paid. 1 TÚG ša qá..."
2,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...,I sold the textiles and received 9 shekels of ...
3,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū...","Seal of Šu-Illil, son of Šu-Kūbum, seal of Ṣil..."
4,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...,From Šukkutum to Ištar-lamassī and Nita-Aššur:...
5,um-ma šu-ta-mu-zi e-lá-a en-um-a-šùr ù lá-ma-s...,"From Šu-Tammuzī, Elaya, Ennam-Aššur and Lamass...",-šu-um a-na šu-IŠTAR i-ša-qú-lu a-ma-kam a-ma-...
6,KIŠIB a-lá-ḫi-im KIŠIB a-li-li KIŠIB a-bi₄-lá ...,"Seal of Ali-ahum, seal of Ali-ilī, seal of Abi...","Seal of Ali-ahum, seal of Ali-ilī, seal of Abī..."
7,a-na É šál-ma-a-šùr a-na a-wa-tim né-ru-ub-ma ...,We entered Šalim-Aššur's house to settle the c...,To the representatives of Šalim-Aššur from Šal...
8,ṭup-pu-um ša 10 ma-na KÙ.BABBAR ša PÚZUR-a-šur...,A tablet about 10 minas of silver of Puzur-Ass...,"Tablets concerning 10 minas of silver, Puzur-A..."
10,2 pí-ri-kà-ni-e šu-ku-tum 2 pí-ri-kà-ni-e en-n...,2 pirikannus: Šukkutum; 2 pirikannus: Enna-Sue...,2 pirikannus: Šu-Kūtum; 2 pirikannus: Enna-Sue...
